# Beca 18 RAG Chatbot

**Course:** Python Programming -- Applied Data Science  
**Task:** Retrieval-Augmented Generation pipeline over the official Beca 18 regulations (PRONABEC).  
**Source document:** Resolucion Directoral Ejecutiva N. 033-2026-MINEDU/VMGI-PRONABEC

Pipeline overview:

```
PDF -> text extraction -> chunking -> embeddings -> ChromaDB
    -> user question -> query embedding -> top-k retrieval
    -> LLM with context -> grounded answer + cited sources
```

## Step 0 -- Setup

Install dependencies and load the Gemini API key from a `.env` file using `python-dotenv`. Never hardcode the key in the notebook. Confirm the environment by printing the loaded package versions.

Get a free API key at: https://aistudio.google.com/app/apikey

In [ ]:
# Uncomment to install in Google Colab (skip locally if conda env already has the packages)
# !pip install pypdf==5.1.0 tiktoken==0.8.0 langchain-text-splitters==0.3.2 google-genai chromadb==0.5.23 ipywidgets==8.1.5 tqdm==4.67.1 python-dotenv==1.0.1

In [1]:
import os
import sys
from pathlib import Path
from importlib.metadata import version

from dotenv import load_dotenv

# --- Load .env from the repo root (parent of notebooks/) ---
ENV_PATH = Path('..').resolve() / '.env'
loaded = load_dotenv(ENV_PATH)

if not loaded:
    raise FileNotFoundError(
        f'.env not found at {ENV_PATH}. '
        'Copy .env.example to .env and add your GEMINI_API_KEY.'
    )

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY or GEMINI_API_KEY == 'your_key_here':
    raise ValueError(
        'GEMINI_API_KEY is missing or still the placeholder. '
        'Edit .env and set a real key from https://aistudio.google.com/app/apikey'
    )

# --- Repository paths ---
REPO_ROOT  = Path('..').resolve()
DATA_DIR   = REPO_ROOT / 'data'
PDF_PATH   = DATA_DIR / 'beca18_reglamento.pdf'
CHROMA_DIR = REPO_ROOT / 'chroma_db_beca18'

if not PDF_PATH.exists():
    raise FileNotFoundError(f'PDF not found at {PDF_PATH}')

# --- Print environment confirmation ---
print('Environment setup OK')
print()
print(f'Python      : {sys.version.split()[0]}')
print(f'Repo root   : {REPO_ROOT}')
print(f'PDF source  : {PDF_PATH.name}  ({PDF_PATH.stat().st_size / 1024:.1f} KB)')
print(f'Chroma path : {CHROMA_DIR}')
print()
print('Loaded API key: GEMINI_API_KEY = ' + GEMINI_API_KEY[:6] + '...' + GEMINI_API_KEY[-4:])
print('  (key is hidden, never printed in full and never hardcoded)')
print()
print('Package versions:')
for pkg in [
    'pypdf', 'tiktoken', 'langchain-text-splitters', 'google-genai',
    'chromadb', 'ipywidgets', 'tqdm', 'python-dotenv',
]:
    try:
        print(f'  {pkg:30s} {version(pkg)}')
    except Exception as e:
        print(f'  {pkg:30s} (not installed: {e})')


Environment setup OK

Python      : 3.11.15
Repo root   : C:\Users\Diego\OneDrive\Documents\GitHub\beca18-rag-chatbot
PDF source  : beca18_reglamento.pdf  (2260.0 KB)
Chroma path : C:\Users\Diego\OneDrive\Documents\GitHub\beca18-rag-chatbot\chroma_db_beca18

Loaded API key: GEMINI_API_KEY = AIzaSy...oGDE
  (key is hidden, never printed in full and never hardcoded)

Package versions:
  pypdf                          5.1.0
  tiktoken                       0.8.0
  langchain-text-splitters       0.3.2
  google-genai                   2.2.0
  chromadb                       0.5.23
  ipywidgets                     8.1.5
  tqdm                           4.67.1
  python-dotenv                  1.0.1


## Step 1 -- PDF Text Extraction

Extract text page by page using `pypdf`. Insert a `[PAGE N]` marker at the start of each page. Apply light cleaning: collapse multiple spaces, remove isolated line breaks, and strip headers/footers if present. Print total character and word counts.

In [2]:
import re
from pypdf import PdfReader

# --- Extract text page by page with markers ---
reader = PdfReader(str(PDF_PATH))
n_pages = len(reader.pages)
print(f'PDF loaded: {n_pages} pages')

raw_pages = []
for i, page in enumerate(reader.pages, start=1):
    txt = page.extract_text() or ''
    raw_pages.append((i, txt))

# --- Light cleaning function ---
def clean_page(text: str) -> str:
    # Strip leading/trailing whitespace
    text = text.strip()

    # Remove common headers/footers (page numbers, copyright lines, etc.)
    # Heuristic: drop very short lines that look like footers/page numbers
    lines = text.split('\n')
    cleaned_lines = []
    for ln in lines:
        s = ln.strip()
        # Drop empty lines and isolated page numbers like "12" or "Pag. 12"
        if not s:
            continue
        if re.fullmatch(r'(p[aá]g(ina)?\.?\s*)?\d{1,3}', s, flags=re.IGNORECASE):
            continue
        cleaned_lines.append(s)

    text = '\n'.join(cleaned_lines)

    # Collapse multiple spaces and tabs
    text = re.sub(r'[ \t]+', ' ', text)

    # Remove isolated line breaks: join lines that don't end with punctuation
    # (keeps paragraph breaks marked by double newlines)
    text = re.sub(r'(?<![.!?:;\n])\n(?!\n)', ' ', text)

    # Collapse 3+ consecutive newlines into 2 (preserve paragraph breaks)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

# --- Apply cleaning and stitch full document with page markers ---
clean_pages = []
for page_num, raw_text in raw_pages:
    cleaned = clean_page(raw_text)
    if cleaned:
        clean_pages.append(f'[PAGE {page_num}]\n{cleaned}')

full_text = '\n\n'.join(clean_pages)

# --- Report ---
total_chars = len(full_text)
total_words = len(full_text.split())

print()
print('=== Extraction Summary ===')
print(f'  Pages processed       : {n_pages}')
print(f'  Pages with content    : {len(clean_pages)}')
print(f'  Total characters      : {total_chars:,}')
print(f'  Total words           : {total_words:,}')
print(f'  Avg chars per page    : {total_chars / max(len(clean_pages), 1):,.0f}')
print()
print('=== Preview (first 600 chars) ===')
print(full_text[:600])
print('...')


PDF loaded: 138 pages

=== Extraction Summary ===
  Pages processed       : 138
  Pages with content    : 138
  Total characters      : 365,895
  Total words           : 55,035
  Avg chars per page    : 2,651

=== Preview (first 600 chars) ===
[PAGE 1]
Resolución Directoral Ejecutiva Nº 033-2026-MINEDU/VMGI-PRONABEC Lima, 24 de febrero de 2026 VISTOS:
El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y Bienestar; el Informe N° 042-2026-MINEDU/VMGI-PRONABEC-OPP de la Oficina de Planeamiento y Presupuesto; el Informe N ° 048-2026-MINEDU/VMGI-PRONABEC-OAJ de la Oficina de Asesoría Jurídica, y;
CONSIDERANDO:
Que, la Ley N° 29837 crea el Programa Nacional de Becas y Crédito Educativo (en adelante, el PRONABEC), a cargo del Ministerio de Edu
...


## Step 2 -- Tokenization and Chunking Justification

Count total tokens using `tiktoken` with the `cl100k_base` encoding. Justify the chunking parameters. Then chunk the cleaned text using LangChain `RecursiveCharacterTextSplitter`.

### Chunking justification: why `chunk_size=400` tokens and `chunk_overlap=60`?

The `gemini-embedding-001` model accepts up to **8,192 tokens per request**, so technically we could embed huge fragments. But choosing a small chunk size has three benefits:

1. **Retrieval precision** -- short chunks (~400 tokens, ~300 words, roughly a paragraph) carry one coherent idea each. When the user asks about a specific topic, the retriever returns a focused fragment instead of a long page with mostly irrelevant content, which improves semantic similarity and reduces noise in the LLM context.

2. **Context window budget for the generator** -- `gemini-2.5-flash` has a generous context window, but every chunk we add as context costs tokens. With chunks of ~400 tokens and `k=5` retrieved chunks, the prompt context is ~2,000 tokens -- leaves plenty of room for the system prompt and the model's response.

3. **8,192 limit is a hard ceiling, not a target** -- using 5% of the limit (400/8192) per chunk guarantees we never approximate the boundary, even with non-ASCII characters or formula-heavy passages that inflate token counts.

The **60-token overlap (~15%)** preserves cross-chunk context: sentences that span chunk boundaries appear in both neighboring chunks, so the retriever cannot miss content that happens to fall on a split. 60 tokens is roughly 2-3 sentences of Spanish legal text, enough to maintain coherence without significant duplication.

In [3]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Token count with cl100k_base ---
enc = tiktoken.get_encoding('cl100k_base')
total_tokens = len(enc.encode(full_text))

print('Token count (cl100k_base):')
print(f'  Total tokens         : {total_tokens:,}')
print(f'  Total characters     : {len(full_text):,}')
print(f'  Avg chars per token  : {len(full_text) / total_tokens:.2f}')
print(f'  Tokens vs 8,192 limit: {total_tokens / 8192:.1f}x the embedding ceiling')
print()

# --- Chunking with RecursiveCharacterTextSplitter ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 400,
    chunk_overlap = 60,
    separators    = ["\n\n", "\n", ". ", " "],
    length_function = lambda t: len(enc.encode(t)),   # measure in TOKENS, not chars
)

chunks_text = splitter.split_text(full_text)

# --- Attach metadata to each chunk ---
import re as _re

def extract_page_number(chunk: str) -> int | None:
    """Return the page number found in [PAGE N] markers within a chunk."""
    m = _re.search(r'\[PAGE (\d+)\]', chunk)
    return int(m.group(1)) if m else None

chunks = []
for idx, txt in enumerate(chunks_text):
    page = extract_page_number(txt)
    metadata = {
        'document' : 'beca18_reglamento.pdf',
        'topic'    : 'Beca 18 -- Reglamento PRONABEC 2026',
        'language' : 'es',
        'page'     : page,
        'chunk_id' : idx,
    }
    chunks.append({'text': txt, 'metadata': metadata})

# --- Report ---
n_chunks   = len(chunks)
char_lens  = [len(c['text']) for c in chunks]
tok_lens   = [len(enc.encode(c['text'])) for c in chunks]

print('Chunking results:')
print(f'  Total chunks               : {n_chunks}')
print(f'  Avg chunk length (chars)   : {sum(char_lens)/n_chunks:,.1f}')
print(f'  Avg chunk length (tokens)  : {sum(tok_lens)/n_chunks:.1f}')
print(f'  Min / Max chunk (tokens)   : {min(tok_lens)} / {max(tok_lens)}')
print()
print('Sample chunk (chunk 0):')
print('-' * 60)
print(chunks[0]['text'][:300] + '...')
print('-' * 60)
print('Metadata:', chunks[0]['metadata'])


Token count (cl100k_base):
  Total tokens         : 102,419
  Total characters     : 365,895
  Avg chars per token  : 3.57
  Tokens vs 8,192 limit: 12.5x the embedding ceiling

Chunking results:
  Total chunks               : 382
  Avg chunk length (chars)   : 979.8
  Avg chunk length (tokens)  : 274.2
  Min / Max chunk (tokens)   : 5 / 399

Sample chunk (chunk 0):
------------------------------------------------------------
[PAGE 1]
Resolución Directoral Ejecutiva Nº 033-2026-MINEDU/VMGI-PRONABEC Lima, 24 de febrero de 2026 VISTOS:
El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y Bienestar; el Informe N° 042-2026-MINED...
------------------------------------------------------------
Metadata: {'document': 'beca18_reglamento.pdf', 'topic': 'Beca 18 -- Reglamento PRONABEC 2026', 'language': 'es', 'page': 1, 'chunk_id': 0}


## Step 3 -- Embeddings

Implement two embedding functions using `gemini-embedding-001` (768 dimensions):

- `embed_documents(texts)` -- uses task type `RETRIEVAL_DOCUMENT` for indexing.
- `embed_query(text)` -- uses task type `RETRIEVAL_QUERY` for search.

Add exponential backoff and retry logic to handle the free-tier rate limit (approximately 60 requests per minute).

In [4]:
import time
import random
from google import genai
from google.genai import types

EMBED_MODEL = 'gemini-embedding-001'
EMBED_DIM   = 768

# --- Gemini client ---
client = genai.Client(api_key=GEMINI_API_KEY)

# --- Exponential backoff retry decorator ---
def with_retry(max_attempts=5, base_delay=1.0, max_delay=60.0):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            attempt = 0
            while True:
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    attempt += 1
                    msg = str(e).lower()
                    # Retry on rate-limit / transient errors
                    is_retryable = any(s in msg for s in [
                        'rate', 'quota', 'resource exhausted', 'unavailable',
                        '429', '503', 'deadline', 'timeout',
                    ])
                    if attempt >= max_attempts or not is_retryable:
                        raise
                    # Exponential backoff with jitter
                    delay = min(base_delay * (2 ** (attempt - 1)), max_delay)
                    delay = delay * (0.5 + random.random())
                    print(f'  [retry] attempt {attempt}/{max_attempts} after {delay:.1f}s -- {type(e).__name__}: {str(e)[:80]}')
                    time.sleep(delay)
        return wrapper
    return decorator


@with_retry(max_attempts=5, base_delay=1.0)
def _embed_batch(texts, task_type: str):
    """Internal: embed a batch of texts with a given task type."""
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=EMBED_DIM,
        ),
    )
    return [e.values for e in result.embeddings]


def embed_documents(texts: list[str], batch_size: int = 5,
                    sleep_between_batches: float = 1.1) -> list[list[float]]:
    """Embed a list of documents with RETRIEVAL_DOCUMENT task type.

    Batches to respect rate limits (~60 RPM free tier).
    """
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        vecs = _embed_batch(batch, task_type='RETRIEVAL_DOCUMENT')
        all_embeddings.extend(vecs)
        # Pace ourselves to stay below 60 RPM on free tier
        if i + batch_size < len(texts):
            time.sleep(sleep_between_batches)
    return all_embeddings


def embed_query(text: str) -> list[float]:
    """Embed a single query with RETRIEVAL_QUERY task type."""
    vecs = _embed_batch([text], task_type='RETRIEVAL_QUERY')
    return vecs[0]


# --- Smoke test on a small sample ---
print('Smoke test: embedding 2 sample documents and 1 query...')
sample_docs = [
    'Los postulantes a la Beca 18 deben cumplir con los requisitos academicos.',
    'El estipendio mensual se otorga para cubrir gastos de manutencion.',
]
doc_vecs   = embed_documents(sample_docs)
query_vec  = embed_query('cuanto es el monto mensual de la beca?')

print()
print(f'Document embeddings : {len(doc_vecs)} vectors')
print(f'  Vector dimension  : {len(doc_vecs[0])}')
print(f'  First 5 values    : {[round(v, 4) for v in doc_vecs[0][:5]]}')
print()
print(f'Query embedding     : 1 vector')
print(f'  Vector dimension  : {len(query_vec)}')
print(f'  First 5 values    : {[round(v, 4) for v in query_vec[:5]]}')

# Verify dimension matches the spec
assert len(doc_vecs[0]) == EMBED_DIM, f'Expected {EMBED_DIM} dims, got {len(doc_vecs[0])}'
assert len(query_vec)   == EMBED_DIM, f'Expected {EMBED_DIM} dims, got {len(query_vec)}'
print()
print('Embedding functions ready. Both task types working at 768 dimensions.')


Smoke test: embedding 2 sample documents and 1 query...

Document embeddings : 2 vectors
  Vector dimension  : 768
  First 5 values    : [-0.0193, 0.0093, 0.0263, -0.0539, 0.0015]

Query embedding     : 1 vector
  Vector dimension  : 768
  First 5 values    : [0.0078, 0.0245, 0.0247, -0.07, -0.0172]

Embedding functions ready. Both task types working at 768 dimensions.


## Step 4 -- Vector Database

Create a persistent ChromaDB collection using cosine distance. Implement idempotent indexing: check whether the collection is already populated before embedding. If it contains documents, skip the embedding step and load the existing collection.

In [5]:
import chromadb
from tqdm.auto import tqdm

COLLECTION_NAME = 'beca18_chunks'

# --- Persistent ChromaDB client ---
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# --- Get or create collection with cosine distance ---
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'},
)

print(f'ChromaDB collection: {COLLECTION_NAME}')
print(f'  Path             : {CHROMA_DIR}')
print(f'  Distance metric  : cosine')
print(f'  Current count    : {collection.count()}')
print()

# --- Idempotent indexing: only embed if empty ---
def sanitize_metadata(md: dict) -> dict:
    """ChromaDB requires primitive metadata values; replace None with empty string."""
    clean = {}
    for k, v in md.items():
        if v is None:
            clean[k] = ''
        elif isinstance(v, (str, int, float, bool)):
            clean[k] = v
        else:
            clean[k] = str(v)
    return clean


if collection.count() == 0:
    print(f'Collection is empty -- embedding {len(chunks)} chunks...')
    print(f'(This may take ~{len(chunks) * 1.1 / 5 / 60:.1f} min at 5 chunks/batch, 1.1s pause)')
    print()

    texts     = [c['text']                     for c in chunks]
    metadatas = [sanitize_metadata(c['metadata']) for c in chunks]
    ids       = [f'chunk_{i:04d}'              for i in range(len(chunks))]

    # Embed in batches with progress bar (uses the embed_documents from Step 3)
    batch_size = 5
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Embedding batches'):
        batch = texts[i:i + batch_size]
        vecs  = _embed_batch(batch, task_type='RETRIEVAL_DOCUMENT')
        embeddings.extend(vecs)
        if i + batch_size < len(texts):
            time.sleep(1.1)

    # Add to ChromaDB
    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas,
    )
    print()
    print(f'Indexed {len(texts)} chunks into the collection.')
else:
    print(f'Collection already populated -- skipping embedding step.')
    print(f'Reusing existing index with {collection.count()} chunks.')

print()
print(f'=== Final collection state ===')
print(f'  Total stored documents: {collection.count()}')

# Show a sample stored item
sample = collection.peek(limit=1)
print()
print('Sample stored chunk:')
print(f'  id       : {sample["ids"][0]}')
print(f'  metadata : {sample["metadatas"][0]}')
print(f'  text     : {sample["documents"][0][:200]}...')


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


ChromaDB collection: beca18_chunks
  Path             : C:\Users\Diego\OneDrive\Documents\GitHub\beca18-rag-chatbot\chroma_db_beca18
  Distance metric  : cosine
  Current count    : 382

Collection already populated -- skipping embedding step.
Reusing existing index with 382 chunks.

=== Final collection state ===
  Total stored documents: 382

Sample stored chunk:
  id       : chunk_0000
  metadata : {'chunk_id': 0, 'document': 'beca18_reglamento.pdf', 'language': 'es', 'page': 1, 'topic': 'Beca 18 -- Reglamento PRONABEC 2026'}
  text     : [PAGE 1]
Resolución Directoral Ejecutiva Nº 033-2026-MINEDU/VMGI-PRONABEC Lima, 24 de febrero de 2026 VISTOS:
El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por la Dirección de Gestió...


## Step 5 -- Semantic Search

Implement `semantic_search(question, k=5)` that:

1. Embeds the question using `embed_query`.
2. Queries the ChromaDB collection for the top-k nearest chunks.
3. Returns a list of dictionaries containing `text`, `metadata`, and `distance`.

Test with one sample question and print the top-3 results with their distances.

In [6]:
def semantic_search(question: str, k: int = 5) -> list[dict]:
    """Embed the question and retrieve top-k nearest chunks from ChromaDB.

    Returns a list of dicts with keys: text, metadata, distance.
    """
    # Embed the query with RETRIEVAL_QUERY task type
    q_vec = embed_query(question)

    # Query ChromaDB
    res = collection.query(
        query_embeddings=[q_vec],
        n_results=k,
    )

    # Format as list of dicts
    results = []
    for doc, md, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        results.append({
            'text'    : doc,
            'metadata': md,
            'distance': float(dist),
        })

    return results


# --- Test with a sample question ---
sample_question = 'Cuales son los requisitos para postular a la Beca 18?'
print(f'Sample question: {sample_question}')
print('=' * 70)

top_results = semantic_search(sample_question, k=3)

for i, r in enumerate(top_results, start=1):
    print()
    print(f'--- Result {i}  (distance = {r["distance"]:.4f}) ---')
    print(f'Page    : {r["metadata"].get("page", "n/a")}')
    print(f'Chunk id: {r["metadata"].get("chunk_id", "n/a")}')
    print(f'Text    : {r["text"][:350]}{"..." if len(r["text"]) > 350 else ""}')

print()
print('=' * 70)
print(f'Returned {len(top_results)} chunks (cosine distance: 0 = identical, 2 = opposite).')


Sample question: Cuales son los requisitos para postular a la Beca 18?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



--- Result 1  (distance = 0.2179) ---
Page    : 102
Chunk id: 289
Text    : [PAGE 102]
N° REQUISITO DOCUMENTO DE ACREDITACIÓN o FORMA DE ACREDITACIÓN documento oficial de la IES donde se acredite que mantiene dicha vacante.
Haber culminado el nivel secundario de la Educación Básica Regular (EBR) o Alternativa (EBA) o Especial (EBE). Los estudios deben ser reconocidos por el Ministerio de Educación.
*En el caso de la Beca 1...

--- Result 2  (distance = 0.2219) ---
Page    : 
Chunk id: 122
Text    : I. Para dimensionar a la población objetivo de Beca 18 se hace uso de la información contenida en SIAGIE entre los años 2020 y 2025. Particularmente, para las personas que estén cursando el último año de educación básica en el 2025, se usará el promedio de notas de sus dos últimos años culminados. En el caso d e los egresados de educación básica en...

--- Result 3  (distance = 0.2236) ---
Page    : 89
Chunk id: 252
Text    : [PAGE 89]
3.2. El concurso financia estudios de pregrado en univ 

## Step 6 -- Grounded Generation

Implement `answer_with_context(question, k=5)` using `gemini-2.5-flash` with a strict system prompt that:

1. Answers EXCLUSIVELY from the retrieved context.
2. Cites the page number when available.
3. Responds with the exact phrase *"The document does not contain information about this topic."* when the context is insufficient.

Test with 5 on-topic questions + 1 off-topic question to verify the model refuses to hallucinate.

In [7]:
LLM_MODEL = 'gemini-2.5-flash'

SYSTEM_PROMPT = """You are a strict assistant for the official Beca 18 regulations (PRONABEC, Peru).

STRICT RULES:
1. Answer EXCLUSIVELY using information from the CONTEXT fragments below. Do NOT use any external or prior knowledge.
2. Never invent, infer beyond, or extrapolate from the context. If a number, deadline, or requirement is not explicitly in the context, do not make it up.
3. ALWAYS cite the page number using [PAGE N] inline whenever you state a fact retrieved from a specific page.
4. If the CONTEXT does not contain enough information to answer the question, you MUST respond with this EXACT phrase and nothing else:
   "The document does not contain information about this topic."
5. Respond in the same language as the user's question (Spanish or English). Be concise, accurate, and professional.
6. Quote literal phrasing from the document when appropriate to preserve legal precision."""


@with_retry(max_attempts=5, base_delay=2.0)
def _generate(question: str, context: str) -> str:
    """Internal: call Gemini with system prompt + retrieved context + question."""
    user_prompt = f"""CONTEXT (fragments retrieved from the Beca 18 regulations):
{context}

USER QUESTION:
{question}"""

    response = client.models.generate_content(
        model=LLM_MODEL,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.1,
        ),
    )
    return (response.text or '').strip()


def answer_with_context(question: str, k: int = 5) -> dict:
    """RAG pipeline: retrieve top-k chunks and generate grounded answer.

    Returns a dict with: answer, sources (list of dicts with text/metadata/distance).
    """
    retrieved = semantic_search(question, k=k)

    # Build context with explicit fragment numbering + page tagging
    context_parts = []
    for i, r in enumerate(retrieved, start=1):
        page = r['metadata'].get('page', '')
        page_tag = f'[PAGE {page}]' if page != '' else '[PAGE n/a]'
        context_parts.append(f'--- Fragment {i} {page_tag} ---\n{r["text"]}')
    context_text = '\n\n'.join(context_parts)

    answer = _generate(question, context_text)
    return {'answer': answer, 'sources': retrieved}


# ====================================================================
# Test: 5 on-topic + 1 off-topic
# ====================================================================
test_questions = [
    ('on-topic',  'Cuales son los requisitos de elegibilidad para postular a la Beca 18?'),
    ('on-topic',  'Cuantas modalidades de Beca 18 existen y cuales son?'),
    ('on-topic',  'Cual es el monto del estipendio mensual otorgado al becario?'),
    ('on-topic',  'Cuales son las obligaciones del becario durante el tiempo de la beca?'),
    ('on-topic',  'En que casos se pierde o se da por concluida la Beca 18?'),
    ('off-topic', 'Cual es la capital de Japon y cuantos habitantes tiene Tokio?'),
]

for kind, q in test_questions:
    print('=' * 78)
    print(f'[{kind.upper()}] {q}')
    print('=' * 78)
    result = answer_with_context(q, k=5)
    print(result['answer'])
    print()
    print('Top sources:')
    for i, s in enumerate(result['sources'][:3], start=1):
        page = s['metadata'].get('page', 'n/a')
        print(f'  {i}. page={page}  distance={s["distance"]:.4f}')
    print()


[ON-TOPIC] Cuales son los requisitos de elegibilidad para postular a la Beca 18?
Los requisitos de elegibilidad para postular a la Beca 18 Ordinaria son los siguientes:

1.  **Edad:** Tener una edad menor de 22 años a la fecha de publicación de las bases [PAGE n/a] o de la Norma Técnica [PAGE 40].
2.  **Condición Socioeconómica:** Clasificación como pobre o pobre extremo según el Sistema de Focalización de Hogares (SISFOH) [PAGE n/a, PAGE 40].
3.  **Rendimiento Académico:**
    *   Tercio superior en los dos últimos grados concluidos de secundaria EBR o EBE o su equivalente en EBA, según corresponda [PAGE 40].
    *   Para las personas que estén cursando el último año de educación básica en el 2025, se usará el promedio de notas de sus dos últimos años culminados [PAGE n/a].
    *   Para los egresados de educación básica en el 2024, se usarán los promedios de notas de su último y penúltimo año de estudios [PAGE n/a].
4.  **Nivel de Estudios:**
    *   Haber culminado el nivel secundari

## Step 7 -- Interactive Chat Interface

Build a chat interface using `ipywidgets` with:

- A text input box for the question.
- "Ask" and "Clear" buttons.
- An integer slider to control the value of `k` (retrieved chunks).
- An output area that displays the answer and an expandable accordion showing the source fragments with their page numbers and distances.

In [8]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# =====================================================================
# Header
# =====================================================================
header = widgets.HTML("""
<div style='background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%);
            padding: 22px; border-radius: 10px; color: white;
            font-family: -apple-system, "Segoe UI", sans-serif; margin-bottom: 14px;'>
    <h2 style='margin:0; font-weight:600;'>Chatbot Normativa Beca 18</h2>
    <p style='margin:8px 0 0 0; opacity:0.92; font-size:0.95em;'>
        Resolucion Directoral Ejecutiva N. 033-2026-MINEDU-VMGI-PRONABEC
    </p>
</div>""")

# =====================================================================
# Output area (shows welcome message until first query)
# =====================================================================
panel = widgets.Output()

with panel:
    display(HTML("""
    <div style='background:#f8fafc; border:1px solid #e2e8f0; border-radius:8px;
                padding:28px; text-align:center; color:#64748b;
                font-style:italic; font-family:-apple-system,"Segoe UI",sans-serif;'>
        Escribe una pregunta sobre la normativa de Beca 18 para iniciar la conversacion.
    </div>"""))

# =====================================================================
# Controls
# =====================================================================
input_query = widgets.Textarea(
    placeholder='Ej: Cuales son los requisitos para postular a la Beca 18?',
    layout=widgets.Layout(width='100%', height='90px'),
)

btn_ask = widgets.Button(
    description='Consultar',
    button_style='primary',
    layout=widgets.Layout(width='160px', height='42px'),
)

btn_clear = widgets.Button(
    description='Limpiar conversacion',
    layout=widgets.Layout(width='180px', height='42px'),
)

slider_k = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description='Fragmentos:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
)

# =====================================================================
# Render the answer + sources accordion
# =====================================================================
def render_answer(question: str, result: dict):
    # Question card
    q_html = f"""
    <div style='background:#fff; padding:16px 18px; border-radius:8px;
                border:1px solid #e2e8f0; margin-bottom:12px;
                font-family:-apple-system,"Segoe UI",sans-serif;'>
        <div style='color:#1e40af; font-weight:600; font-size:0.78em;
                    text-transform:uppercase; letter-spacing:0.4px;'>PREGUNTA</div>
        <div style='font-size:1.02em; color:#0f172a; margin-top:6px;'>{question}</div>
    </div>"""

    # Answer card
    answer_html = f"""
    <div style='background:#fff; padding:18px; border-radius:8px;
                border:1px solid #e2e8f0; margin-bottom:12px;
                font-family:-apple-system,"Segoe UI",sans-serif;'>
        <div style='color:#15803d; font-weight:600; font-size:0.78em;
                    text-transform:uppercase; letter-spacing:0.4px;'>RESPUESTA</div>
        <div style='font-size:1.0em; color:#0f172a; margin-top:10px;
                    line-height:1.65; white-space:pre-wrap;'>{result['answer']}</div>
    </div>"""

    display(HTML(q_html + answer_html))

    # Sources accordion (one entry per retrieved chunk)
    accordion_items = []
    titles = []
    for i, s in enumerate(result['sources'], start=1):
        page = s['metadata'].get('page', 'n/a') or 'n/a'
        dist = s['distance']
        chunk_id = s['metadata'].get('chunk_id', 'n/a')

        body = widgets.HTML(f"""
        <div style='padding:8px 4px; font-family:-apple-system,"Segoe UI",sans-serif;
                    font-size:0.9em; color:#1e293b; line-height:1.6;'>
            <div style='margin-bottom:8px; color:#64748b;'>
                <b>Pagina:</b> {page} &nbsp; | &nbsp;
                <b>Chunk id:</b> {chunk_id} &nbsp; | &nbsp;
                <b>Distancia coseno:</b> {dist:.4f}
            </div>
            <div style='background:#f8fafc; padding:10px; border-left:3px solid #3b82f6;
                        border-radius:4px; white-space:pre-wrap;'>{s['text']}</div>
        </div>""")
        accordion_items.append(body)
        titles.append(f'Fragmento {i}  -  pag. {page}  -  dist. {dist:.4f}')

    accordion = widgets.Accordion(children=accordion_items, selected_index=None)
    for i, t in enumerate(titles):
        accordion.set_title(i, t)

    display(HTML("""
    <div style='font-family:-apple-system,"Segoe UI",sans-serif;
                color:#64748b; font-weight:600; font-size:0.78em;
                text-transform:uppercase; letter-spacing:0.4px;
                margin-top:6px; margin-bottom:6px;'>
        FRAGMENTOS FUENTE (clic para expandir)
    </div>"""))
    display(accordion)

# =====================================================================
# Event handlers
# =====================================================================
def on_ask(_):
    q = input_query.value.strip()
    if not q:
        with panel:
            clear_output()
            display(HTML("<div style='color:#b91c1c; padding:12px;'>Escribe una pregunta primero.</div>"))
        return

    with panel:
        clear_output()
        display(HTML("<div style='color:#3b82f6; padding:12px;'>Buscando fragmentos y consultando Gemini...</div>"))

    try:
        result = answer_with_context(q, k=slider_k.value)
    except Exception as e:
        with panel:
            clear_output()
            display(HTML(f"<div style='color:#b91c1c; padding:12px;'>Error: {e}</div>"))
        return

    with panel:
        clear_output()
        render_answer(q, result)


def on_clear(_):
    input_query.value = ''
    with panel:
        clear_output()
        display(HTML("""
        <div style='background:#f8fafc; border:1px solid #e2e8f0; border-radius:8px;
                    padding:28px; text-align:center; color:#64748b;
                    font-style:italic; font-family:-apple-system,"Segoe UI",sans-serif;'>
            Escribe una pregunta sobre la normativa de Beca 18 para iniciar la conversacion.
        </div>"""))


btn_ask.on_click(on_ask)
btn_clear.on_click(on_clear)

# =====================================================================
# Layout
# =====================================================================
controls_row = widgets.HBox([btn_ask, btn_clear, slider_k])
ui = widgets.VBox([header, panel, input_query, controls_row])
display(ui)
